# 03. Feature Engineering

> **Краткое резюме**: Объединяем профильные и поведенческие данные в единую матрицу признаков. Encoding категориальных переменных, log-трансформация, нормализация, обработка пропусков. Результат — `data/feature_matrix.parquet`.

**Предпосылки**: `01_etl_profile` и `02_etl_behavior` выполнены.

---

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

OUTPUT_DIR = os.path.abspath('./data')
# Spark для чтения Parquet-директорий
try:
    _ = spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName('LookAlike_FE').enableHiveSupport().getOrCreate()

SPARK_DIR = f'file://{OUTPUT_DIR}'

---
## 1. Загрузка данных

In [ ]:
profiles_sdf = spark.read.parquet(f'{SPARK_DIR}/client_profiles.parquet')
behavior_sdf = spark.read.parquet(f'{SPARK_DIR}/behavioral_features.parquet')

# Merge в Spark, затем toPandas
merged_sdf = profiles_sdf.join(behavior_sdf, on='client_uk', how='inner')
print(f'Merged (Spark): {merged_sdf.count():,} rows')

df = merged_sdf.toPandas()
df = df.set_index('client_uk')
print(f'Loaded to Pandas: {df.shape}')
df.head()

---
## 2. Производные признаки

In [ ]:
# Direction ratio: outflow / total (0 = только получает, 1 = только платит)
df['direction_ratio'] = df['total_outflow'] / df['total_amount'].replace(0, np.nan)
df['direction_ratio'] = df['direction_ratio'].fillna(0.5)

# Средний месячный оборот
df['avg_monthly_amount'] = df['total_amount'] / df['active_months'].replace(0, 1)

# Тренд оборота: (вторая половина - первая) / первая
df['amount_growth'] = (
    (df['amount_second_half'] - df['amount_first_half'])
    / df['amount_first_half'].replace(0, np.nan)
).clip(-5, 5).fillna(0)

# Рост контрагентов
df['cp_growth'] = (
    (df['cp_second_half'] - df['cp_first_half'])
    / df['cp_first_half'].replace(0, np.nan)
).clip(-5, 5).fillna(0)

# Коэффициент вариации суммы транзакции
df['tx_amount_cv'] = df['std_tx_amount'] / df['avg_tx_amount'].replace(0, np.nan)
df['tx_amount_cv'] = df['tx_amount_cv'].fillna(0)

print('Derived features created.')
df[['direction_ratio', 'avg_monthly_amount', 'amount_growth', 'cp_growth', 'tx_amount_cv']].describe()

---
## 3. Encoding категориальных признаков

In [ ]:
# ОКВЭД: top-N + 'OTHER'
OKVED_TOP_N = 20
if 'sparkokved_ccode' in df.columns:
    okved = df['sparkokved_ccode'].fillna('UNKNOWN')
    top_okveds = okved.value_counts().head(OKVED_TOP_N).index.tolist()
    okved_mapped = okved.where(okved.isin(top_okveds), 'OTHER')
    okved_dummies = pd.get_dummies(okved_mapped, prefix='okved')
    df = pd.concat([df, okved_dummies], axis=1)
    print(f'OKVED: {OKVED_TOP_N} top codes + OTHER → {okved_dummies.shape[1]} dummy columns')

# Регион: top-N + 'OTHER'
REGION_TOP_N = 20
if 'addrref_region_uk' in df.columns:
    region = df['addrref_region_uk'].fillna(-1).astype(int).astype(str)
    top_regions = region.value_counts().head(REGION_TOP_N).index.tolist()
    region_mapped = region.where(region.isin(top_regions), 'OTHER')
    region_dummies = pd.get_dummies(region_mapped, prefix='region')
    df = pd.concat([df, region_dummies], axis=1)
    print(f'Region: {REGION_TOP_N} top + OTHER → {region_dummies.shape[1]} dummy columns')

# Модельный сегмент (если доступен)
if 'srvpackagesegment_ccode' in df.columns:
    seg = df['srvpackagesegment_ccode'].fillna('UNKNOWN')
    seg_dummies = pd.get_dummies(seg, prefix='mseg')
    df = pd.concat([df, seg_dummies], axis=1)
    print(f'Model segment → {seg_dummies.shape[1]} dummy columns')
else:
    print('Model segment not available — skipped')

# Тип клиента
if 'client_type_name' in df.columns:
    ctype_dummies = pd.get_dummies(df['client_type_name'].fillna('UNKNOWN'), prefix='ctype')
    df = pd.concat([df, ctype_dummies], axis=1)
    print(f'Client type → {ctype_dummies.shape[1]} dummy columns')

---
## 4. Log-трансформация и нормализация

In [ ]:
# Монетарные признаки: log1p трансформация
MONETARY_COLS = [
    'total_amount', 'total_outflow', 'total_inflow',
    'avg_tx_amount', 'avg_monthly_amount',
    'registry_authcapital_amt',
]
# Добавляем опциональные колонки
for col in ['avg_balance', 'max_balance', 'product_total_amt', 'revenue_rur_amt']:
    if col in df.columns:
        MONETARY_COLS.append(col)

for col in MONETARY_COLS:
    if col in df.columns:
        df[col] = np.log1p(df[col].fillna(0).clip(lower=0))

print(f'Log1p applied to {len([c for c in MONETARY_COLS if c in df.columns])} monetary columns')

In [ ]:
# Формируем финальный набор числовых фичей
NUMERIC_FEATURES = [
    # Поведенческие
    'total_amount', 'total_outflow', 'total_inflow',
    'tx_count', 'avg_tx_amount', 'std_tx_amount',
    'unique_counterparties', 'unique_payees', 'unique_payers',
    'active_months',
    'direction_ratio', 'avg_monthly_amount',
    'amount_growth', 'cp_growth', 'tx_amount_cv',
]
# Опциональные
for col in ['avg_balance', 'max_balance', 'min_balance', 'avg_balance_30d',
            'product_count', 'product_type_count', 'product_total_amt',
            'registry_authcapital_amt', 'revenue_rur_amt']:
    if col in df.columns:
        NUMERIC_FEATURES.append(col)

# Dummy-колонки (ОКВЭД, регион, сегмент, тип)
DUMMY_COLS = [c for c in df.columns if c.startswith(('okved_', 'region_', 'mseg_', 'ctype_'))]

FEATURE_COLS = NUMERIC_FEATURES + DUMMY_COLS
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

print(f'Feature columns: {len(FEATURE_COLS)} ({len(NUMERIC_FEATURES)} numeric + {len(DUMMY_COLS)} dummy)')

In [ ]:
# Заполнение пропусков и нормализация
X = df[FEATURE_COLS].fillna(0).copy()

# StandardScaler для числовых
numeric_in_X = [c for c in NUMERIC_FEATURES if c in X.columns]
scaler = StandardScaler()
X[numeric_in_X] = scaler.fit_transform(X[numeric_in_X])

print(f'Feature matrix: {X.shape}')
X.describe()

---
## 5. Сохранение

In [ ]:
import pickle

# Сохраняем матрицу фичей
feature_path = os.path.join(OUTPUT_DIR, 'feature_matrix.parquet')
X.to_parquet(feature_path)
print(f'Feature matrix saved: {feature_path}')

# Сохраняем метаданные (scaler, список фичей, маппинг ОКВЭД)
meta = {
    'feature_cols': FEATURE_COLS,
    'numeric_features': numeric_in_X,
    'dummy_cols': DUMMY_COLS,
    'scaler': scaler,
}
meta_path = os.path.join(OUTPUT_DIR, 'feature_meta.pkl')
with open(meta_path, 'wb') as f:
    pickle.dump(meta, f)
print(f'Metadata saved: {meta_path}')

# Сохраняем полный df (с сырыми + derived + dummy колонками) для анализа
full_path = os.path.join(OUTPUT_DIR, 'full_client_data.parquet')
# Сохраняем только сериализуемые колонки
save_cols = [c for c in df.columns if df[c].dtype.name != 'object' or c in [
    'client_name', 'client_type_name', 'clientcounterparty_flag',
    'sparkokved_ccode', 'addrref_city_name', 'reg_city_name',
    'srvpackagesegment_ccode', 'clientchange_date', 'client_rel_date',
    'entrepreneur_flag', 'client_active_flag',
]]
df[save_cols].to_parquet(full_path)
print(f'Full data saved: {full_path}')

---

## Глоссарий

| Термин | Описание |
|--------|----------|
| **direction_ratio** | Доля исходящих платежей: 0 = только получает, 1 = только платит |
| **amount_growth** | Рост оборота: (2-я половина - 1-я) / 1-я. Clipped [-5, 5] |
| **cp_growth** | Рост числа контрагентов по той же формуле |
| **tx_amount_cv** | Коэффициент вариации суммы транзакции (std / mean). Высокий = неравномерные платежи |
| **log1p** | Логарифм (1+x). Сжимает распределение монетарных переменных |
| **StandardScaler** | Нормализация к среднему=0, std=1. Необходима для K-Means и distance-based методов |
| **One-hot (dummy)** | Кодирование категориальной переменной: 1 колонка на каждое значение |

---

**Следующий шаг**: `04_segmentation.ipynb`.